# Sesión 1 · Qué es MCP y tu primer servidor remoto

**Curso MCP · servidores remotos** — notebook 1 de 4

Al final de esta sesión tendrás **tu propio servidor MCP desplegado en Cloud Run**, con URL
pública, exponiendo un dataset de BigQuery. Y habrás visto el protocolo por dentro, sin SDK
de por medio, antes de dejar que una librería te lo esconda.

| Bloque | Minutos |
|---|:--:|
| Qué es MCP y por qué existe | 20 |
| El protocolo por dentro | 25 |
| Despliegue a Cloud Run | 25 |
| Tools: servidor y cliente | 40 |

> **Antes de empezar** necesitas un proyecto de Google Cloud con facturación activada.
> Si llegas sin eso, esta sesión no te va a dar tiempo.

## 1. Qué es MCP y por qué existe

Tienes datos en BigQuery. Quieres que un modelo pueda consultarlos.

Sin un protocolo común, escribes un integrador para Claude, otro para ChatGPT, otro para
Cursor, otro para tu agente propio. Cuatro integraciones para el mismo dato, y cada vez que
cambia el esquema, cuatro sitios que tocar.

**Model Context Protocol es el contrato que rompe esa multiplicación.** Escribes un servidor
una vez, y cualquier cliente que hable MCP puede usarlo. La analogía útil: MCP es a los
modelos lo que un driver es a un sistema operativo.

### Las tres piezas

| Pieza | Qué es | En este curso |
|---|---|---|
| **Servidor** | Publica capacidades: acciones, datos, plantillas | Lo que vas a escribir, sobre BigQuery |
| **Cliente** | Habla el protocolo, uno por servidor conectado | El SDK de Python en este notebook |
| **Host** | La aplicación donde vive la conversación | Claude Desktop, ChatGPT, Cursor… |

El **host** es quien manda: decide qué servidores conecta, qué le enseña al modelo y qué
permisos concede. El servidor **ofrece**, nunca impone. Esa asimetría explica casi todas las
decisiones de diseño del protocolo, y conviene tenerla presente desde el principio.

### Por qué remoto y no local

Un servidor MCP puede correr como proceso local en la máquina del usuario. En este curso no
vamos a hacer eso **en ningún momento**, y no es un capricho:

- **Distribución.** Un servidor remoto se comparte con una URL. Uno local hay que instalarlo
  en cada máquina, con su runtime y sus dependencias.
- **Actualizaciones.** Despliegas una vez y todos tus usuarios tienen la versión nueva.
- **Control de acceso.** Es donde vive el OAuth de verdad, y donde tú decides quién ve qué.
- **Es donde están tus datos.** BigQuery ya está en la nube; bajar el servidor a un portátil
  para hablar con la nube es un rodeo.

Todo lo que aprendas aquí sirve tal cual en producción.

## 2. El protocolo por dentro

Antes de tocar el SDK, vamos a hablar con un servidor MCP **a mano**. Diez minutos de
incomodidad que ahorran meses de tratar la librería como una caja negra.

MCP es **JSON-RPC 2.0 sobre HTTP**. Nada más. Un `POST` con un cuerpo JSON, y una respuesta
JSON. La revisión que usamos es la `2026-07-28`.

In [ ]:
!pip install --quiet "mcp==2.0.0" httpx

> ### ⚠️ Necesitas una URL para esta parte
>
> Para hablar con un servidor MCP hace falta un servidor MCP. Todavía no has desplegado el
> tuyo —eso es el bloque 3—, así que aquí usamos uno de referencia.
>
> **Tienes dos opciones:**
>
> 1. **El servidor de referencia del curso.** Quien imparte la sesión lo despliega antes de
>    clase (`gcloud run deploy` con este mismo repositorio) y reparte la URL. Pégala en la
>    celda siguiente.
> 2. **Si estás haciendo el notebook por tu cuenta** y no tienes esa URL: salta al **bloque 3**,
>    despliega tu servidor, y vuelve aquí con tu propia URL. El contenido de este bloque no
>    depende de que el servidor sea de nadie en concreto.
>
> **Para quien imparte:** despliega el servidor de referencia *antes* de la primera sesión y
> ten la URL escrita en la pizarra. Si los veinte alumnos tienen que desplegar antes de ver un
> solo mensaje del protocolo, la sesión se convierte en soporte técnico de `gcloud`.

In [ ]:
import httpx, json

# ← EDITAR: URL del servidor de referencia (te la da quien imparte la sesión),
#           o la de tu propio servidor si ya has hecho el bloque 3.
URL = "https://PON-AQUI-LA-URL.run.app/mcp"

peticion = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "server/discover",
    "params": {
        "_meta": {
            "io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientInfo": {"name": "notebook-curso", "version": "1.0"},
        }
    },
}

respuesta = httpx.post(
    URL,
    json=peticion,
    headers={
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
        # Cabeceras que la revisión 2026-07-28 exige en cada POST
        "Mcp-Method": "server/discover",
        "Mcp-Name": "curso",
    },
    timeout=30,
)
print(respuesta.status_code)
print(json.dumps(respuesta.json(), indent=2, ensure_ascii=False)[:1200])

### Qué acabas de ver

`server/discover` es una llamada obligatoria del protocolo, y responde a todo lo que un
cliente necesita saber antes de empezar:

- **`supportedVersions`** — qué revisiones habla este servidor.
- **`capabilities`** — qué ofrece: tools, resources, prompts, extensiones.
- **`_meta.io.modelcontextprotocol/serverInfo`** — quién es.
- **`ttlMs` y `cacheScope`** — cuánto puede cachear el cliente esta respuesta.

Fíjate en lo que **no** hay: ninguna sesión, ningún identificador de conexión. La revisión
`2026-07-28` es **stateless**. Cada petición se explica sola y lleva su propia versión y sus
propias capacidades en `_meta`.

Eso tiene una consecuencia directa para nosotros: **sin estado de sesión, un servidor MCP
escala en horizontal sin afinidad de sesión**. Es exactamente lo que Cloud Run necesita para
repartir peticiones entre instancias sin pensárselo.

In [ ]:
# Lo mismo, pero preguntando qué acciones ofrece.
peticion = {
    "jsonrpc": "2.0", "id": 2, "method": "tools/list",
    "params": {"_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28"}},
}
r = httpx.post(URL, json=peticion, headers={
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "Mcp-Method": "tools/list", "Mcp-Name": "curso",
}, timeout=30).json()

for t in r["result"]["tools"]:
    print(f"- {t['name']}: {t.get('description','')[:70]}")

> **Ejercicio (3 min).** Cambia `protocolVersion` a `"1999-01-01"` y vuelve a lanzar la
> celda. El servidor responde con `UnsupportedProtocolVersionError` y te dice qué versiones
> sí soporta. Así negocia un cliente que no sabe con quién habla.

## 3. Tu servidor en Cloud Run

Ahora el tuyo. Un único despliegue que ya contiene el código de las cuatro sesiones.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROYECTO = "pon-aqui-tu-project-id"  # ← EDITAR
REGION = "europe-west1"
SERVICIO = "curso-mcp"

!gcloud config set project {PROYECTO}

In [ ]:
!gcloud services enable run.googleapis.com cloudbuild.googleapis.com bigquery.googleapis.com

In [ ]:
!git clone --quiet https://github.com/noelserdna/curso-mcp /content/curso-mcp
%cd /content/curso-mcp
!ls

In [ ]:
# Un único despliegue para todo el curso. Tarda un par de minutos la primera vez.
!gcloud run deploy {SERVICIO} \
  --source . \
  --region {REGION} \
  --allow-unauthenticated \
  --set-env-vars CURSO_MCP_DATASET=austin_bikeshare \
  --quiet

In [ ]:
MI_URL = !gcloud run services describe {SERVICIO} --region {REGION} --format="value(status.url)"
MI_URL = MI_URL[0] + "/mcp"
print("Tu servidor:", MI_URL)

> **`--allow-unauthenticated` es deliberado y temporal.** Ahora mismo cualquiera con la URL
> puede usar tu servidor y facturarte las consultas. En la sesión 3 lo cerramos. Que quede
> claro que un servidor abierto a internet **no** es un estado aceptable para algo que toca
> datos.

## 4. Tools: la primitiva central

Un **tool** es una acción que el modelo puede invocar. Es la primitiva que decide si tu
servidor sirve para algo.

Así se declara uno en el SDK de Python (esto ya está en `curso_mcp/server.py`):

```python
@mcp.tool(
    title="Describir tabla",
    description=(
        "Devuelve el esquema de una tabla: columnas, tipos y descripción. "
        "Úsalo antes de escribir una consulta para no inventar nombres de columna."
    ),
)
def describir_tabla(tabla: str) -> list[dict[str, Any]]:
    return bq.describir_tabla(tabla)
```

Tres cosas que hace el SDK por ti y conviene saber que hace:

1. **El esquema de entrada sale de las anotaciones de tipo.** `tabla: str` se convierte en
   un JSON Schema que el modelo lee para saber qué mandarte.
2. **El de salida, del tipo de retorno**, y el resultado viaja además como
   `structuredContent`, no solo como texto.
3. **Las excepciones se convierten en errores de protocolo** con `is_error`, en vez de tumbar
   la conexión.

In [ ]:
import asyncio
from mcp import Client

async def explorar():
    async with Client(MI_URL) as c:
        lista = await c.list_tools()
        for t in lista.tools:
            print(f"· {t.name}")
        print()
        r = await c.call_tool("listar_tablas", {})
        print(r.content[0].text)

await explorar()

> **Ojo con `await` en Colab.** El notebook ya corre dentro de un bucle de eventos, así que
> puedes usar `await` directamente en la celda. En un script normal harías
> `asyncio.run(explorar())`.

In [ ]:
async def esquema_de(tabla):
    async with Client(MI_URL) as c:
        r = await c.call_tool("describir_tabla", {"tabla": tabla})
        # El resultado estructurado, además del texto
        return r.structured_content

await esquema_de("bikeshare_trips")

### Diseñar tools que el modelo entienda

Aquí es donde se gana o se pierde la partida, y no es cuestión de código sino de redacción.

**El esquema de tus tools entra en el contexto del modelo en cada conversación.** Cien tools
mal descritos son cien tools que compiten por atención y por tokens.

- **La descripción se escribe para el modelo, no para el usuario.** Di *cuándo* usar el tool,
  no solo qué hace. Compara: «Devuelve el esquema» contra «Úsalo antes de escribir una
  consulta para no inventar nombres de columna».
- **Nombres explícitos.** `describir_tabla` gana a `get_meta`.
- **Pocos y buenos.** Por debajo de unas quince acciones, uno por acción. Por encima, hay un
  patrón mejor (`search` + `execute`) que verás en el material de ampliación.
- **Orden estable en `tools/list`.** El SDK lo mantiene; eso permite al cliente cachear y
  mejora los aciertos de caché de prompt.

## Ejercicios

1. **Un tool nuevo.** Añade a `curso_mcp/server.py` un tool `contar_filas(tabla)` que
   devuelva el número de filas. Redespliega y llámalo desde aquí.
2. **Rompe una descripción.** Cambia la de `describir_tabla` por «Devuelve datos». Conecta
   Claude Desktop a tu URL y observa cómo empeoran sus decisiones sobre cuándo llamarlo.
3. **Sin SDK.** Reproduce la llamada a `listar_tablas` con `httpx` puro, como en el bloque 2.

## En la próxima sesión

Las otras primitivas: **resources** para datos que trae el host, **prompts** para workflows
enlatados, **interacción** para cuando el servidor necesita preguntarte algo a mitad de una
llamada, y **progreso** para operaciones que tardan.